<a href="https://colab.research.google.com/github/RayanAhmadKhan/RAG/blob/main/RAG2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installing required libraries for processing text, vector searches, and API handling
!pip install -q sentence-transformers faiss-cpu google-generativeai

In [ ]:
import os

# 1. Establishing project directory layout as per specifications
os.makedirs("documents", exist_ok=True)
os.makedirs("faiss_index", exist_ok=True)

# 2. Defining multilingual mock data (English and Urdu)
documents_data = {
    "documents/admission.txt": """
    Admission Policy 2026:
    Applications for the Fall semester open on June 1st and close definitively on August 15th.
    Applicants must secure a minimum of 60% marks in their Higher Secondary School Certificate (HSSC) to qualify for undergraduate programs.
    Late submissions incur a penalty fee of $50.

    داخلہ پالیسی 2026:
    خریف سمسٹر کے لیے درخواستیں 1 جون سے شروع ہوں گی اور 15 اگست کو بند ہو جائیں گی۔
    انڈرگریجویٹ پروگراموں کے اہل ہونے کے لیے درخواست گزاروں کے انٹر میڈیٹ میں کم از کم 60 فیصد نمبر ہونا لازمی ہیں۔
    تاخیر سے جمع کرانے پر 50 ڈالر جرمانہ فیس ہوگی۔
    """,

    "documents/hostel.txt": """
    Hostel Allotment & Accommodation Rules:
    Hostel rooms are distributed on a strict 'first-come, first-served' policy.
    The total annual fee for the hostel residency is $500. This structural fee covers mess utilities, high-speed internet, and electricity.

    ہاسٹل الاٹمنٹ اور رہائشی قواعد:
    ہاسٹل کے کمرے سخت 'پہلے آؤ، پہلے پاؤ' کی بنیاد پر تقسیم کیے جاتے ہیں۔
    ہاسٹل کی رہائش کی کل سالانہ فیس 500 ڈالر ہے۔ اس فیس میں میس، تیز رفتار انٹرنیٹ اور بجلی کے اخراجات شامل ہیں۔
    """,

    "documents/scholarship.txt": """
    Scholarship & Financial Aid Guidelines:
    Merit-based scholarships covering 100% tuition fees are distributed to the top 5% of students in each department.
    Need-based financial aid requires the submission of official tax returns or signed income certificates of the legal guardian.

    اسکالرشپ اور مالی امداد کی ہدایات:
    100 فیصد ٹیوشن فیس کا احاطہ کرنے والی میرٹ پر مبنی اسکالرشپس ہر شعبے کے سرفہرست 5 فیصد طلبہ میں تقسیم کی جاتی ہیں۔
    ضرورت کی بنیاد پر مالی امداد کے لیے قانونی سرپرست کے آفیشل ٹیکس گوشوارے یا آمدنی کا تصدیق شدہ سرٹیفکیٹ جمع کروانا لازمی ہے۔
    """
}

# 3. Write data to disk using UTF-8 encoding to preserve Urdu script text integrity
for path, content in documents_data.items():
    with open(path, "w", encoding="utf-8") as file:
        file.write(content.strip())

print("✅ Data directory 'documents/' populated with English and Urdu source files.")

✅ Data directory 'documents/' populated with English and Urdu source files.


In [13]:
import os
import glob
import pickle
import numpy as np
import faiss
import google.generativeai as genai
from google.colab import userdata
from sentence_transformers import SentenceTransformer

class MultilingualRAGPipeline:
    def __init__(self, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        """Initializes embedding models, indices, and connects to the cloud LLM infrastructure."""
        print("Initializing Multilingual Embedding Model...")
        self.embedding_model = SentenceTransformer(model_name)
        self.index = None
        self.metadata = []

        # Authenticate with Gemini API via Colab Secret Utility
        try:
            self.api_key = userdata.get('GEMINI_API_KEY')
            genai.configure(api_key=self.api_key)

            # FIXED: Updated to an active, supported production model endpoint
            self.llm = genai.GenerativeModel('gemini-2.5-flash')
            print("Successfully authenticated with Google Gemini API (gemini-2.5-flash).")
        except Exception as e:
            print("\n❌ CRITICAL ERROR: Could not find 'GEMINI_API_KEY' in your Colab Secrets.")
            print("Please add your key via the key icon on the left sidebar before executing.\n")

    def chunk_document(self, text, max_chars=400, overlap=50):
        """Splits text documents smoothly based on line breaks to safeguard text context."""
        lines = text.split('\n')
        chunks = []
        current_chunk = ""

        for line in lines:
            if not line.strip():
                continue
            if len(current_chunk) + len(line) < max_chars:
                current_chunk += line + " "
            else:
                chunks.append(current_chunk.strip())
                current_chunk = line + " "
        if current_chunk:
            chunks.append(current_chunk.strip())
        return chunks

    def build_and_save_vector_store(self, docs_dir="documents", output_dir="faiss_index"):
        """Reads local files, builds embeddings, and commits index matrices to storage files."""
        raw_chunks = []
        self.metadata = []

        file_paths = glob.glob(os.path.join(docs_dir, "*.txt"))
        if not file_paths:
            raise FileNotFoundError(f"No text files found inside directory: '{docs_dir}'")

        # Parsing local text repositories
        for path in file_paths:
            filename = os.path.basename(path)
            with open(path, "r", encoding="utf-8") as f:
                text_content = f.read()
                chunks = self.chunk_document(text_content)
                for chunk in chunks:
                    raw_chunks.append(chunk)
                    self.metadata.append({"source": filename, "text": chunk})

        print(f"Processing and Vectorizing {len(raw_chunks)} text segments...")
        embeddings = self.embedding_model.encode(raw_chunks, show_progress_bar=True)

        # Build standard FAISS L2 Distance Index matrix
        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(np.array(embeddings).astype("float32"))

        # Write to local file structures as demanded by requirements
        faiss.write_index(self.index, os.path.join(output_dir, "index.faiss"))
        with open(os.path.join(output_dir, "metadata.pkl"), "wb") as f:
            pickle.dump(self.metadata, f)

        print(f"✅ FAISS Vector Storage compiled successfully at '{output_dir}/'.")

    def load_vector_store(self, output_dir="faiss_index"):
        """Loads static vectors back into runtime execution space."""
        self.index = faiss.read_index(os.path.join(output_dir, "index.faiss"))
        with open(os.path.join(output_dir, "metadata.pkl"), "rb") as f:
            self.metadata = pickle.load(f)
        print("✅ Vector database re-loaded into memory.")

    def query_rag_system(self, user_query, top_k=2):
        """Executes retrieval similarity matrix lookups and infers structured prompt results."""
        if self.index is None:
            self.load_vector_store()

        # Generate target vector tracking metrics
        query_vector = self.embedding_model.encode([user_query]).astype("float32")
        distances, indices = self.index.search(query_vector, top_k)

        retrieved_contexts = []
        sources = set()
        for idx in indices[0]:
            if idx < len(self.metadata):
                match = self.metadata[idx]
                retrieved_contexts.append(match["text"])
                sources.add(match["source"])

        context_block = "\n".join(retrieved_contexts)

        # Language detection logic embedded inside instruction tuning structures
        prompt = f"""
        You are a precise educational helper system. Answer the query contextually using only the provided facts.
        If the query is in Urdu, write your explanation perfectly in Urdu.
        If the query is in English, write your explanation perfectly in English.
        If the question cannot be accurately supported from the context block, return: "Information is not present in the local database."

        Context Block:
        {context_block}

        User Query:
        {user_query}

        Response:
        """

        # Call Google Gemini API
        response = self.llm.generate_content(prompt)
        return response.text, list(sources)

# Run structural processing routines immediately upon executing this cell
pipeline = MultilingualRAGPipeline()
pipeline.build_and_save_vector_store()

Initializing Multilingual Embedding Model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Successfully authenticated with Google Gemini API (gemini-2.5-flash).
Processing and Vectorizing 6 text segments...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ FAISS Vector Storage compiled successfully at 'faiss_index/'.


In [14]:
print("=========================================================================")
print("🌐 ONLINE MULTILINGUAL RAG SYSTEM ACTIVE (English / اردو) 🌐")
print("=========================================================================")
print("Instructions: Type your question below and press enter. Type 'exit' to quit.")
print("Try asking: 'What is the cutoff percentage for admission?' or 'ہاسٹل کی فیس کتنی ہے؟'")
print("-------------------------------------------------------------------------\n")

while True:
    user_input = input("👤 Ask a Question (or type 'exit'): ")
    if user_input.strip().lower() == 'exit':
        print("Stopping RAG Session. Goodbye!")
        break

    if not user_input.strip():
        continue

    try:
        print("🔍 Searching local FAISS indices and generating response via Gemini API...")
        answer, utilized_sources = pipeline.query_rag_system(user_input)

        print("\n🤖 System Response:")
        print(f"{answer.strip()}")
        print(f"📦 [Retrieved from local files: {', '.join(utilized_sources)}]")
        print("-" * 73 + "\n")

    except Exception as error:
        print(f"An execution issue occurred: {error}\n")

🌐 ONLINE MULTILINGUAL RAG SYSTEM ACTIVE (English / اردو) 🌐
Instructions: Type your question below and press enter. Type 'exit' to quit.
Try asking: 'What is the cutoff percentage for admission?' or 'ہاسٹل کی فیس کتنی ہے؟'
-------------------------------------------------------------------------

👤 Ask a Question (or type 'exit'): ہاسٹل کی فیس کتنی ہے؟
🔍 Searching local FAISS indices and generating response via Gemini API...

🤖 System Response:
ہاسٹل کی رہائش کی کل سالانہ فیس 500 ڈالر ہے۔
📦 [Retrieved from local files: hostel.txt, scholarship.txt]
-------------------------------------------------------------------------

👤 Ask a Question (or type 'exit'): when will admission open for fall ?
🔍 Searching local FAISS indices and generating response via Gemini API...

🤖 System Response:
Applications for the Fall semester will open on June 1st.
📦 [Retrieved from local files: admission.txt]
-------------------------------------------------------------------------

👤 Ask a Question (or type 'e

In [25]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

def generate_linkedin_portfolio_visualization(pipeline, user_query):
    """
    Generates and saves a high-resolution 2D map of the multilingual vector space,
    showing exactly how an English or Urdu query clusters next to database documents.
    """
    print("Preparing vector space visualization map...")

    # 1. Extract texts from the metadata
    if not pipeline.metadata:
        pipeline.load_vector_store()

    texts = [item["text"] for item in pipeline.metadata]
    sources = [item["source"] for item in pipeline.metadata]

    # 2. Re-generate or fetch embeddings for the map
    embeddings = pipeline.embedding_model.encode(texts)
    query_embedding = pipeline.embedding_model.encode([user_query])

    # Combine embeddings to fit PCA transformation uniformly
    all_embeddings = np.vstack([embeddings, query_embedding])

    # 3. Reduce dimensionality from 384 dimensions to 2D using PCA
    pca = PCA(n_components=2, random_state=42)
    coords_2d = pca.fit_transform(all_embeddings)

    doc_coords = coords_2d[:-1]
    query_coord = coords_2d[-1]

    # 4. Initialize Plot Styling
    plt.figure(figsize=(11, 7), dpi=150)
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

    # Map sources to unique colors and markers
    unique_sources = list(set(sources))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    markers = ['o', 's', '^']
    source_to_color = {src: colors[i % len(colors)] for i, src in enumerate(unique_sources)}
    source_to_marker = {src: markers[i % len(markers)] for i, src in enumerate(unique_sources)}

    # Plot document chunks grouped by file source
    for src in unique_sources:
        idx = [i for i, s in enumerate(sources) if s == src]
        plt.scatter(
            doc_coords[idx, 0], doc_coords[idx, 1],
            label=f"Source Context: {src}",
            c=source_to_color[src], marker=source_to_marker[src],
            s=120, alpha=0.75, edgecolors='black'
        )

    # 5. Execute retrieval algorithm to find the exact top matches to draw trajectories
    query_vec = np.array(query_embedding).astype("float32")
    distances, indices = pipeline.index.search(query_vec, k=2)

    # Plot the User Query vector location as a striking glowing star
    plt.scatter(
        query_coord[0], query_coord[1],
        label=f"User Query Vector",
        c='#e377c2', marker='*', s=350,
        edgecolors='black', linewidths=1.5, zorder=5
    )

    # Draw retrieval trajectory mapping paths
    for match_idx in indices[0]:
        if match_idx < len(doc_coords):
            plt.plot(
                [query_coord[0], doc_coords[match_idx, 0]],
                [query_coord[1], doc_coords[match_idx, 1]],
                c='#d62728', linestyle='--', linewidth=1.5, alpha=0.6
            )

    # Add textual annotation labels to the graph vector coordinates
    plt.text(query_coord[0] + 0.02, query_coord[1] + 0.02, "Query Vector Target Location",
             fontsize=9, weight='bold', bbox=dict(facecolor='white', alpha=0.8, boxstyle='round,pad=0.3'))

    # Annotate a few data clusters dynamically
    for i in [0, len(doc_coords)//2]:
        plt.text(doc_coords[i, 0] + 0.02, doc_coords[i, 1] + 0.02, f"Chunk #{i}", fontsize=7, alpha=0.7)

    # 6. Titles, Legends and Save Mechanics
    plt.title("🎯 Multilingual Vector Space & Semantic Search Trajectory Map", fontsize=14, pad=15, weight='bold')
    plt.xlabel("Principal Component Matrix X Axis Dimension", fontsize=10)
    plt.ylabel("Principal Component Matrix Y Axis Dimension", fontsize=10)
    plt.legend(loc="upper right", frameon=True, facecolor='white', edgecolor='none')
    plt.tight_layout()

    # Save chart file directly to your workspace storage panel
    #output_filename = "linkedin_rag_visualization.png"
    #plt.savefig(output_filename, bbox_inches='tight', dpi=300)
    #plt.show()

    #print(f"\n✅ Success! Chart saved to your Google Colab files workspace as '{output_filename}'.")
    #print("Download this image to add it directly into your GitHub README and LinkedIn post updates.")

# --- EXECUTE DEMO TEST RUN ---
# Replace this string with any text or Urdu phrase to instantly trace it visually!
#test_query = "ہاسٹل کی فیس کتنی ہے؟"
#generate_linkedin_portfolio_visualization(pipeline, test_query)

In [26]:
import IPython
from google.colab import output


try:

    output.eval_js('google.colab.kernel.proxyPort(8000)')
    print("Polishing active notebook session state...")


    if hasattr(IPython.get_ipython(), 'kernel'):
        kernel = IPython.get_ipython().kernel
        if hasattr(kernel, 'shell') and 'widgets' in kernel.shell.user_ns:
            del kernel.shell.user_ns['widgets']

    print("🎉 Memory state optimized! The hidden widget metadata has been cleared from this session.")
except Exception as e:
    print("Session state handled automatically.")




Polishing active notebook session state...
🎉 Memory state optimized! The hidden widget metadata has been cleared from this session.
